# Lab 09-07 · The Bipolar Junction Transistor

Companion to chapter [09-07](../../chapters/09-07-bipolar-junction-transistor.md). Handout: [README.md](README.md).

The transistor is the chapter's NPN: $A_E$ = 10 µm², $W_B$ = 0.1 µm, $N_B$ = 10¹⁸, $N_E$ = 10²⁰ ($W_E$ = 0.2 µm, Hu's 95 meV narrowing) and $N_C$ = 10¹⁷ cm⁻³, at 300 K.

In [1]:
import math
import numpy as np
from scipy.integrate import solve_ivp, trapezoid
from scipy.optimize import brentq, minimize_scalar
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
NI = 1e10
EPS = 11.7 * 8.8541878128e-14
AE, WB, NB, NE, WE, NC = 1e-7, 1e-5, 1e18, 1e20, 2e-5, 1e17
mun = lambda N: 92 + 1268 / (1 + (N / 1.3e17) ** 0.91)
mup = lambda N: 48 + 402 / (1 + (N / 6.3e16) ** 0.76)
DB, DE = VT * mun(NB), VT * mup(NE)
IS = Q * AE * DB * NI**2 / (WB * NB)
BETA = (DB * WE * NE) / (DE * WB * NB) / math.exp(0.095 / VT)
print(f"I_S = {IS:.4e} A, ideal beta = {BETA:.2f}")
assert abs(IS - 1.091e-18) / 1.091e-18 < 5e-4 and abs(BETA - 27.0) < 0.05

I_S = 1.0906e-18 A, ideal beta = 26.99


## Part 1 — Gummel plot and current gain

$I_C$ includes high-level injection, through $n'(N_B + n') = n_i^2e^{V_{BE}/V_T}$. $I_B$ is the ideal emitter-injection current plus depletion-region recombination $I_{r0}(e^{V_{BE}/2V_T} - 1)$, with $I_{r0} = qn_iWA_E/2\tau_0$ for an assumed $W$ = 30 nm and $\tau_0$ = 0.1 µs.

*Predict first:* over how many decades of $I_C$ does the gain stay within a factor of 2 of its peak?

In [2]:
IR0 = Q * NI * 3e-6 * AE / (2 * 1e-7)
def ic(V):
    n_ = (-NB + math.sqrt(NB**2 + 4 * NI**2 * math.exp(V / VT))) / 2
    return Q * AE * DB * n_ / WB
def ib(V):
    return IS / BETA * math.exp(V / VT) + IR0 * (math.exp(V / (2 * VT)) - 1)
beta = lambda V: ic(V) / ib(V)
pk = minimize_scalar(lambda V: -beta(V), bounds=(0.5, 1.0), method="bounded", options={"xatol": 1e-10})
bpk = -pk.fun
Vhi = brentq(lambda V: beta(V) - bpk / 2, pk.x, 1.3)
Vlo = brentq(lambda V: beta(V) - bpk / 2, 0.2, pk.x)
print(f"peak beta {bpk:.2f} at I_C = {ic(pk.x)*1e3:.3f} mA")
print(f"beta halves at I_C = {ic(Vlo)*1e9:.3f} nA and {ic(Vhi)*1e3:.2f} mA: {math.log10(ic(Vhi)/ic(Vlo)):.1f} decades")
V = np.linspace(0.3, 1.05, 400)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].semilogy(V, [ic(x) for x in V], label="I_C"); ax[0].semilogy(V, [ib(x) for x in V], label="I_B")
ax[0].set_xlabel("V_BE (V)"); ax[0].legend()
ax[1].semilogx([ic(x) for x in V], [beta(x) for x in V]); ax[1].set_xlabel("I_C (A)"); ax[1].set_ylabel("beta")
plt.tight_layout(); plt.show()
assert abs(bpk - 26.6) < 0.05 and abs(ic(Vhi) * 1e3 - 11.2) < 0.05 and abs(ic(Vlo) * 1e9 - 3.66) < 0.005

peak beta 26.63 at I_C = 0.049 mA
beta halves at I_C = 3.658 nA and 11.19 mA: 6.5 decades


## Part 2 — The Early effect from the moving depletion edge

The neutral base width is 0.1 µm at $V_{CB}$ = 2 V and changes as the base-side depletion edge moves. $V_{BE}$ is held fixed, so $I_C \propto 1/W_B$.

*Predict first:* is the extrapolated intercept larger or smaller than the local $Q'_B/C'_{jC}$ = 31.4 V?

In [3]:
def xp_base(VCB):
    Vbi = VT * math.log(NB * NC / NI**2)
    W = math.sqrt(2 * EPS * (Vbi + VCB) * (1 / NB + 1 / NC) / Q)
    return W * NC / (NB + NC)
def ic_rel(VCB):
    return 1 / (WB + xp_base(2.0) - xp_base(VCB))
h = 1e-4
VA_local = ic_rel(2.0) / ((ic_rel(2 + h) - ic_rel(2 - h)) / (2 * h))
VBE0 = VT * math.log(1e-3 / IS)
vce = np.linspace(1.0, 3.0, 201)
icv = np.array([ic_rel(v - VBE0) for v in vce]) / ic_rel(2.0) * 1e-3
slope, icpt = np.polyfit(vce, icv, 1)
print(f"local V_A at V_CB = 2 V: {VA_local:.2f} V;  extrapolated intercept over V_CE 1-3 V: {icpt/slope:.2f} V")
assert abs(VA_local - 31.4) < 0.05 and abs(icpt / slope - 24.5) < 0.05

local V_A at V_CB = 2 V: 31.37 V;  extrapolated intercept over V_CE 1-3 V: 24.47 V


## Part 3 — Base transit time with a graded SiGe base

The electron flux $\mu_n\mathcal En - D_B\,dn/dx$ is constant, with $\mu_n\mathcal E = D_B\eta/W_B$ and $\eta = \Delta E_{gB}/kT$. Integrate the profile back from $n(W_B) = 0$ for unit flux; the stored charge is the transit time.

*Predict first:* does 0.2 eV of grading halve the transit time of 0.1 eV?

In [4]:
def tau_b(dEg, W=WB, D=DB):
    eta = dEg / VT
    x = np.linspace(W, 0, 4001)
    sol = solve_ivp(lambda xx, y: [(D * eta / W * y[0] - 1.0) / D], [W, 0], [0.0], t_eval=x, rtol=1e-11, atol=1e-30)
    return trapezoid(sol.y[0][::-1], x[::-1])
grads = np.linspace(0, 0.2, 21)
taus = [tau_b(g) for g in grads]
for g in (0.0, 0.1, 0.2):
    print(f"grading {g:.1f} eV: tau_B = {tau_b(g)*1e12:.3f} ps")
plt.plot(grads, np.array(taus) * 1e12); plt.xlabel("grading (eV)"); plt.ylabel("tau_B (ps)"); plt.show()
assert abs(tau_b(0.0) * 1e12 - 7.35) < 0.005 and abs(tau_b(0.1) * 1e12 - 2.84) < 0.005

grading 0.0 eV: tau_B = 7.345 ps
grading 0.1 eV: tau_B = 2.837 ps
grading 0.2 eV: tau_B = 1.654 ps


## Part 4 — f_T against collector current

Here $f_T = 1/2\pi[\tau_B + \tau_C + (C_{jE} + C_{jC})V_T/I_C]$, with $C_{jE}$ = 20 fF and $C_{jC}$ = 10.2 fF. Above $J_K = qN_Cv_{\text{sat}}$ the base pushes out by $W_{dC}(1 - J_K/J)$, a simple illustrative Kirk model.

*Predict first (the chapter's guided exercise):* where does $f_T$ peak, and why?

In [5]:
Vbi = VT * math.log(NB * NC / NI**2)
W = math.sqrt(2 * EPS * (Vbi + 2) * (1 / NB + 1 / NC) / Q)
WDC = W * NB / (NB + NC)
CJC = EPS / W * 2 * AE
CJE = 20e-15
VSAT = 1.53e9 * 300**-0.87
JK = Q * NC * VSAT
def fT(I, graded=False):
    J = I / AE
    push = WDC * (1 - JK / J) if J > JK else 0.0
    tb = tau_b(0.1 if graded else 0.0, W=WB + push)
    return 1 / (2 * math.pi * (tb + (WDC - push) / (2 * VSAT) + (CJE + CJC) * VT / I))
Is_ = np.sort(np.append(np.geomspace(1e-5, 1e-1, 801), JK * AE))   # include the Kirk current
fu = np.array([fT(i) for i in Is_])
k = int(np.argmax(fu))
print(f"f_T at 1 mA: {fT(1e-3)/1e9:.2f} GHz (uniform), {fT(1e-3, True)/1e9:.2f} GHz (graded)")
print(f"uniform-base peak {fu[k]/1e9:.2f} GHz at {Is_[k]*1e3:.2f} mA; Kirk current {JK*AE*1e3:.2f} mA")
plt.semilogx(Is_ * 1e3, fu / 1e9); plt.xlabel("I_C (mA)"); plt.ylabel("f_T (GHz)"); plt.show()
assert abs(fT(1e-3) / 1e9 - 17.7) < 0.05 and abs(fT(1e-3, True) / 1e9 - 35.5) < 0.05
assert abs(fu[k] / 1e9 - 19.3) < 0.05 and abs(Is_[k] * 1e3 - 17.2) < 0.05

f_T at 1 mA: 17.71 GHz (uniform), 35.53 GHz (graded)
uniform-base peak 19.29 GHz at 17.15 mA; Kirk current 17.15 mA


## Questions

Answer the discussion questions in the [lab handout](README.md), then try the optional extension (a gradual Kirk model).